# **Phase 3: API Development and Deployment**

## **Executive Summary**
- We developed a production-ready REST API using FastAPI to serve our best ML model (SVM with StandardScaler)
- The API includes feature engineering, input validation with Pydantic, and proper error handling
- A React frontend provides an intuitive user interface for risk prediction
- Deployed to Render with automated CI/CD from GitHub
- Key validation: API predictions match direct model predictions exactly
- Live demo: https://sp-frontend-qtk3.onrender.com

## **1. Tech Stack**
- **Backend**: FastAPI (Python) - Fast, modern, automatic API documentation
- **Frontend**: React + Vite + Tailwind CSS
- **Deployment**: Render (Backend Web Service + Frontend Static Site)
- **CI/CD**: GitHub auto-deploy

## **2. Setup**

In [13]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import requests
import json

# Add project root to path if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

### **Load the Best Model**

From Phase 2, our best model was: **SVM with StandardScaler and class weights**

In [14]:
# Load the trained model and scaler
MODEL_PATH = '../models/best_model__standard_scaled__svm__class_weight.pkl'
SCALER_PATH = '../models/scaler_standard.pkl'

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print(f"Model type: {type(model).__name__}")
print(f"Scaler type: {type(scaler).__name__}")
print(f"\nScaler expects these features:")
print(scaler.feature_names_in_)

Model type: Pipeline
Scaler type: StandardScaler

Scaler expects these features:
['bmi' 'hypertension' 'pulse_pressure' 'cigarettes_per_day'
 'total_cholesterol' 'glucose' 'heart_rate' 'age_group_code']


## **3. Feature Engineering in the API**

### **Challenge**
Our model was trained on engineered features (e.g., `pulse_pressure = sysBP - diaBP`), but users should only input raw clinical data.

### **Solution**
The API automatically calculates derived features from user inputs.

### **Input → Feature Transformation**

The API uses **snake_case** naming convention:
- `systolic_bp`, `diastolic_bp` (not `sysBP`, `diaBP`)
- `heart_rate` (not `heartRate`)
- `cigarettes_per_day` (not `cigsPerDay`)
- `total_cholesterol` (not `totChol`)
- `hypertension` (not `prevalentHyp`)

In [15]:
def calculate_age_group_code(age: int) -> int:
    """
    Calculate age group code based on age ranges.
    
    Age ranges (must match training data):
        0: <45 years
        1: 45-54 years
        2: 55-64 years
        3: 65+ years
    
    Args:
        age: Patient age in years
    
    Returns:
        int: Age group code (0-3)
    """
    if age < 45:
        return 0
    elif age < 55:
        return 1
    elif age < 65:
        return 2
    else:
        return 3


def prepare_features(patient_data):
    """
    Transform raw patient data into model features.
    This is exactly what the API does internally.
    
    Input: Raw clinical measurements (API format with snake_case)
    Output: Engineered features (what model expects)
    """
    
    # Derived feature: pulse pressure
    pulse_pressure = patient_data['systolic_bp'] - patient_data['diastolic_bp']
    
    # Age group encoding
    age_group_code = calculate_age_group_code(patient_data['age'])
    
    # Create feature dictionary with EXACT names the scaler expects
    features = {
        'bmi': float(patient_data['bmi']),
        'hypertension': int(patient_data['hypertension']),
        'pulse_pressure': float(pulse_pressure),
        'cigarettes_per_day': float(patient_data['cigarettes_per_day']),
        'total_cholesterol': float(patient_data['total_cholesterol']),
        'glucose': float(patient_data['glucose']),
        'heart_rate': float(patient_data['heart_rate']),
        'age_group_code': int(age_group_code)
    }
    
    return pd.DataFrame([features])

# Example usage
example_patient = {
    'age': 50,
    'systolic_bp': 120,
    'diastolic_bp': 80,
    'bmi': 25.0,
    'heart_rate': 70,
    'total_cholesterol': 200,
    'glucose': 90,
    'cigarettes_per_day': 10,
    'hypertension': 0  # 0=No, 1=Yes
}

features_df = prepare_features(example_patient)
print("\nEngineered features:")
print(features_df)
print(f"\nFeature names: {features_df.columns.tolist()}")
print(f"\nScaler expects: {scaler.feature_names_in_.tolist()}")


Engineered features:
    bmi  hypertension  pulse_pressure  cigarettes_per_day  total_cholesterol  \
0  25.0             0            40.0                10.0              200.0   

   glucose  heart_rate  age_group_code  
0     90.0        70.0               1  

Feature names: ['bmi', 'hypertension', 'pulse_pressure', 'cigarettes_per_day', 'total_cholesterol', 'glucose', 'heart_rate', 'age_group_code']

Scaler expects: ['bmi', 'hypertension', 'pulse_pressure', 'cigarettes_per_day', 'total_cholesterol', 'glucose', 'heart_rate', 'age_group_code']


### **Direct Model Prediction (Baseline)**

First, let's make a prediction directly using the loaded model. This will be our ground truth for API validation.

In [17]:
# Prepare features
features_df = prepare_features(example_patient)

# Scale features (model was trained on scaled data)
features_scaled = scaler.transform(features_df)

# IMPORTANT: Model is a Pipeline, needs DataFrame with column names
features_scaled_df = pd.DataFrame(
    features_scaled, 
    columns=features_df.columns
)

# Make prediction
prediction = model.predict(features_scaled_df)[0]
probability = model.predict_proba(features_scaled_df)[0][1]  # Probability of class 1 (disease)

print(f"Direct Model Prediction:")
print(f"  Prediction: {prediction} ({'Risk' if prediction == 1 else 'No Risk'})")
print(f"  Probability: {probability:.4f}")
risk_level = 'High' if probability >= 0.6 else 'Medium' if probability >= 0.3 else 'Low'
print(f"  Risk Level: {risk_level}")

Direct Model Prediction:
  Prediction: 0 (No Risk)
  Probability: 0.0985
  Risk Level: Low


## **4. API Architecture**

### **Backend Structure**
```
backend/
├── api/
│   ├── main.py                 # FastAPI app, CORS, endpoints
│   ├── core/
│   │   └── config.py          # Configuration settings
│   ├── schemas/
│   │   └── prediction.py      # Pydantic models for validation
│   └── services/
│       ├── model_service.py   # Model loading & prediction
│       └── feature_service.py # Feature engineering
├── main.py                     # Entry point
├── requirements.txt
└── Dockerfile
```

### **Key API Endpoints**

#### **1. Health Check: `GET /health`**
```json
{
  "status": "healthy",
  "model_loaded": true
}
```

#### **2. Prediction: `POST /predict`**
**Request (API format with snake_case):**
```json
{
  "age": 50,
  "systolic_bp": 120,
  "diastolic_bp": 80,
  "bmi": 25.0,
  "heart_rate": 70,
  "total_cholesterol": 200,
  "glucose": 90,
  "cigarettes_per_day": 10,
  "hypertension": 0
}
```

**Response:**
```json
{
  "prediction": 0,
  "prediction_label": "Low Risk",
  "probability": 0.2345,
  "risk_level": "Low"
}
```

### **Input Validation with Pydantic**

The API automatically validates all inputs:
- Data types (int, float)
- Required fields
- Value ranges (e.g., age 18-120, BMI 15-60)
- Returns clear error messages if invalid

## **5. API Testing: Validation Against Direct Predictions**

### **Do API predictions match direct model predictions?**

This ensures the API correctly implements:
1. Feature engineering
2. Feature scaling
3. Model inference

In [19]:
def test_api_prediction(api_url, patient_data):
    """
    Test API prediction and compare with direct model prediction.
    """
    try:
        # Make API request
        response = requests.post(
            f"{api_url}/predict",
            json=patient_data,
            timeout=30
        )
        
        if response.status_code == 200:
            api_result = response.json()
            
            # Direct model prediction
            features_df = prepare_features(patient_data)
            features_scaled = scaler.transform(features_df)
            features_scaled_df = pd.DataFrame(features_scaled, columns=features_df.columns)
            direct_prediction = model.predict(features_scaled_df)[0]
            direct_probability = model.predict_proba(features_scaled_df)[0][1]
            
            # Compare results
            print("\n" + "="*60)
            print("API vs Direct Model Comparison")
            print("="*60)
            print(f"\nAPI Prediction:")
            print(f"  Prediction: {api_result['prediction']}")
            print(f"  Probability: {api_result['probability']:.4f}")
            print(f"  Risk Level: {api_result['risk_level']}")
            
            print(f"\nDirect Model Prediction:")
            print(f"  Prediction: {direct_prediction}")
            print(f"  Probability: {direct_probability:.4f}")
            
            # Validation
            prediction_match = api_result['prediction'] == direct_prediction
            probability_diff = abs(api_result['probability'] - direct_probability)
            probability_match = probability_diff < 0.0001
            
            print(f"\nValidation Results:")
            print(f"  Prediction Match: {'YES' if prediction_match else 'NO'}")
            print(f"  Probability Match: {'YES' if probability_match else 'NO'}")
            print(f"  Probability Difference: {probability_diff:.6f}")
            
            if prediction_match and probability_match:
                print(f"\nSUCCESS: API predictions match direct model predictions!")
            else:
                print(f"\nWARNING: Predictions don't match!")
                
            return api_result
        else:
            print(f"API Error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except requests.exceptions.ConnectionError:
        print(f"Could not connect to API at {api_url}")
        print(f"The API might be sleeping (Render cold start). Try opening {api_url}/health in browser first.")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

### **Production API Testing**

Test the deployed API on Render:

In [20]:
# Test against PRODUCTION API
PRODUCTION_API_URL = "https://sp-backend-qkjs.onrender.com"

# Health check first
print("Production API Health Check:")
try:
    health_response = requests.get(f"{PRODUCTION_API_URL}/health", timeout=30)
    if health_response.status_code == 200:
        health_data = health_response.json()
        print(json.dumps(health_data, indent=2))
    else:
        print(f"Health check failed: {health_response.status_code}")
except Exception as e:
    print(f"Could not reach API: {e}")
    print("The API might be sleeping. Try opening https://sp-backend-qkjs.onrender.com/health in browser first.")

# Test prediction
print("\nTesting production API prediction...")
test_api_prediction(PRODUCTION_API_URL, example_patient)

Production API Health Check:
{
  "status": "healthy",
  "model_loaded": true
}

Testing production API prediction...

API vs Direct Model Comparison

API Prediction:
  Prediction: 0
  Probability: 0.0985
  Risk Level: Low

Direct Model Prediction:
  Prediction: 0
  Probability: 0.0985

Validation Results:
  Prediction Match: YES
  Probability Match: YES
  Probability Difference: 0.000000

SUCCESS: API predictions match direct model predictions!


{'prediction': 0,
 'prediction_label': 'Low Risk',
 'probability': 0.09847876878779159,
 'risk_level': 'Low'}

### **Test Multiple Cases**

Let's test several patient profiles to ensure consistency:

In [21]:
test_cases = [
    {
        "name": "Low Risk Profile",
        "data": {
            'age': 30,
            'systolic_bp': 110,
            'diastolic_bp': 70,
            'bmi': 22.0,
            'heart_rate': 65,
            'total_cholesterol': 180,
            'glucose': 85,
            'cigarettes_per_day': 0,
            'hypertension': 0
        }
    },
    {
        "name": "High Risk Profile",
        "data": {
            'age': 65,
            'systolic_bp': 160,
            'diastolic_bp': 95,
            'bmi': 32.0,
            'heart_rate': 85,
            'total_cholesterol': 280,
            'glucose': 120,
            'cigarettes_per_day': 20,
            'hypertension': 1
        }
    },
    {
        "name": "Medium Risk Profile",
        "data": {
            'age': 50,
            'systolic_bp': 135,
            'diastolic_bp': 85,
            'bmi': 27.0,
            'heart_rate': 72,
            'total_cholesterol': 220,
            'glucose': 95,
            'cigarettes_per_day': 5,
            'hypertension': 0
        }
    }
]

# Test each case
results = []
for test_case in test_cases:
    print(f"\n{'='*60}")
    print(f"Testing: {test_case['name']}")
    print(f"{'='*60}")
    result = test_api_prediction(PRODUCTION_API_URL, test_case['data'])
    if result:
        results.append({
            'name': test_case['name'],
            'risk_level': result['risk_level'],
            'probability': result['probability']
        })

# Summary
print("\n" + "="*60)
print("Test Summary")
print("="*60)
for r in results:
    print(f"{r['name']:20s} | Risk: {r['risk_level']:8s} | Prob: {r['probability']:.4f}")


Testing: Low Risk Profile

API vs Direct Model Comparison

API Prediction:
  Prediction: 0
  Probability: 0.0415
  Risk Level: Low

Direct Model Prediction:
  Prediction: 0
  Probability: 0.0415

Validation Results:
  Prediction Match: YES
  Probability Match: YES
  Probability Difference: 0.000000

SUCCESS: API predictions match direct model predictions!

Testing: High Risk Profile

API vs Direct Model Comparison

API Prediction:
  Prediction: 1
  Probability: 0.6060
  Risk Level: High

Direct Model Prediction:
  Prediction: 1
  Probability: 0.6060

Validation Results:
  Prediction Match: YES
  Probability Match: YES
  Probability Difference: 0.000000

SUCCESS: API predictions match direct model predictions!

Testing: Medium Risk Profile

API vs Direct Model Comparison

API Prediction:
  Prediction: 0
  Probability: 0.1005
  Risk Level: Low

Direct Model Prediction:
  Prediction: 0
  Probability: 0.1005

Validation Results:
  Prediction Match: YES
  Probability Match: YES
  Probabili

## **6. Deployment**

### **Local Deployment with Docker**

We used Docker Compose to run the application locally before cloud deployment.

**Structure:**
```
docker-compose.yml
├── Backend (Python 3.11, FastAPI, Port 8000)
└── Frontend (Node 18, React, Port 3000)
```

**Process:**
```bash
# 1. Build and start services
docker-compose up --build

# 2. Access locally
# - Frontend: http://localhost:3000
# - Backend: http://localhost:8000

# 3. Stop services
docker-compose down
```

**Key configurations:**
- Backend: Models mounted as volume (`./models:/app/models`)
- Frontend: Environment variable for API URL (`VITE_API_URL=http://localhost:8000`)
- Networking: Services communicate via Docker internal network

---

### **Cloud Deployment with Render**

Once Docker deployment worked, we deployed to Render for public access.

**Configuration:** Blueprint (`render.yaml`)
```yaml
services:
  backend:  Python 3.11, builds from backend/, copies models
  frontend: Static site, builds React app, serves from dist/
```

**Process:**
```
git push → Render detects changes → Auto-build → Deploy → Live URLs
```

**URLs:**
- Frontend: https://sp-frontend-qtk3.onrender.com
- Backend: https://sp-backend-qkjs.onrender.com

**Key difference from Docker:**
- Docker: Private (localhost), fast, always running
- Render: Public (HTTPS), cold starts (~30s after 15min inactivity), auto-deploys from git

## **7. Conclusions phase 3**

### **What We Achieved**

**Functional API**
- FastAPI backend serving SVM model
- Automatic feature engineering (pulse pressure, age groups)
- Input validation with Pydantic
- Proper error handling

**User-Friendly Frontend**
- React interface with intuitive form
- Real-time predictions
- Color-coded risk levels (Low/Medium/High)
- Responsive design

**Production Deployment**
- Deployed on Render (free tier)
- CI/CD from GitHub
- Automatic HTTPS
- Health monitoring

**Validated Accuracy**
- API predictions match direct model predictions exactly
- Tested across multiple patient profiles
- Feature engineering correctly implemented

### **Links**

- 🌐 **Live Demo**: https://sp-frontend-qtk3.onrender.com
- 🔗 **API**: https://sp-backend-qkjs.onrender.com
- 📚 **API Docs**: https://sp-backend-qkjs.onrender.com/docs (Swagger UI)
- 💻 **GitHub**: https://github.com/maria165sa/SP-project-group-D